# 33 · Mixture Boundary Model: Unified Core + Conditional Residuals

This notebook upgrades the unified-vs-specialist comparison from Notebook 32 into a simple **mixture boundary** model.

Previous notebooks suggested:
- a unified boundary captures a large part of the zeta–GUE signal
- specialist boundaries can improve performance in some condition systems
- the best picture is not “unified only” or “specialist only”
- instead, the boundary looks like a **shared core plus condition-specific corrections**

Now we ask:

> Can we explicitly model the boundary as a unified core plus a conditional residual mixture?

## Model families

We compare three strategies:
- **Unified core**: one pooled boundary across low + high conditions
- **Specialist mixture**: separate low / high specialist boundaries
- **Mixture model**: unified core blended with specialist residuals through a simple condition-aware gate

## Condition systems

We test:
- **entropy**: low entropy + high entropy
- **unevenness**: low unevenness + high unevenness

## Tasks

For each system, we compare:
- **zeta vs GUE**
- **zeta vs Poisson**

## Main outputs

- mixture vs unified vs specialist phase-gap bars
- mixture-gain heatmaps
- overlap comparison curves with bootstrap intervals
- compact mixture summary

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Window statistics and feature builder

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_windows_and_features(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    stats = {
        "entropy": np.array([window_entropy(w) for w in windows_n]),
        "unevenness": np.array([unevenness(w) for w in windows_n]),
        "ratio_mean": np.array([ratio_mean(w) for w in windows_n]),
        "symmetry": np.array([reversal_symmetry_score(w) for w in windows_n]),
    }

    if feature_family == "minimal":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["ratio_mean"][i]]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "full":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["symmetry"][i], stats["ratio_mean"][i], ratio_std(windows_n[i])]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(f"Unknown feature family: {feature_family}")

    return windows_n, stats, X

def apply_condition_mask(stats, condition_name):
    if condition_name == "low_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] <= thr
    if condition_name == "high_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] > thr
    if condition_name == "low_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] <= thr
    if condition_name == "high_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] > thr
    raise ValueError(condition_name)

## RBF boundary helpers

In [ ]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std, mean, std

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    med = np.median(D[D > 0])
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :])**2, axis=2)
    return np.exp(-gamma * D2)

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def fit_rbf_boundary(X0_train, X1_train):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]

    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))

    Xtr_std, _, mean, std = standardize_train_test(X_train, X_train)
    prototypes = choose_prototypes(Xtr_std, n_proto=min(20, len(Xtr_std)))
    gamma = estimate_rbf_gamma(Xtr_std)
    R_train = rbf_features(Xtr_std, prototypes, gamma)
    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)

    return {
        "mean": mean,
        "std": std,
        "prototypes": prototypes,
        "gamma": gamma,
        "w": w,
    }

def boundary_scores(model, X0_test, X1_test):
    m = min(len(X0_test), len(X1_test))
    X0_test = X0_test[:m]
    X1_test = X1_test[:m]
    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test)])

    Xte_std = (X_test - model["mean"]) / model["std"]
    R_test = rbf_features(Xte_std, model["prototypes"], model["gamma"])
    scores = decision_function_logistic(R_test, model["w"])
    probs = predict_proba_logistic(R_test, model["w"])
    return y_test, scores, probs

def evaluate_scores(y_test, scores, probs):
    preds = (probs >= 0.5).astype(int)
    acc = float(np.mean(preds == y_test))
    scores0 = scores[y_test == 0]
    scores1 = scores[y_test == 1]
    overlap = overlap_coefficient_from_hist(scores0, scores1, bins=30)
    return {"accuracy": acc, "overlap": overlap}

## Mixture boundary model

In [ ]:
def evaluate_rbf_boundary(model, X0_test, X1_test):
    y_test, scores, probs = boundary_scores(model, X0_test, X1_test)
    return evaluate_scores(y_test, scores, probs)

def blend_probs(p_unified, p_specialist, alpha):
    return (1.0 - alpha) * p_unified + alpha * p_specialist

def evaluate_mixture_model(unified_model, specialist_model, X0_test, X1_test, alpha=0.5):
    y_u, _, pu = boundary_scores(unified_model, X0_test, X1_test)
    y_s, _, ps = boundary_scores(specialist_model, X0_test, X1_test)
    assert np.array_equal(y_u, y_s)
    p_mix = blend_probs(pu, ps, alpha)
    eps = 1e-8
    scores_mix = np.log(np.clip(p_mix, eps, 1-eps) / np.clip(1-p_mix, eps, 1-eps))
    return evaluate_scores(y_u, scores_mix, p_mix)

def evaluate_unified_specialist_mixture(X0_low, X1_low, X0_high, X1_high, alpha=0.5):
    m = min(len(X0_low), len(X1_low), len(X0_high), len(X1_high))
    X0_low = X0_low[:m]
    X1_low = X1_low[:m]
    X0_high = X0_high[:m]
    X1_high = X1_high[:m]

    unified = fit_rbf_boundary(np.vstack([X0_low, X0_high]), np.vstack([X1_low, X1_high]))
    spec_low = fit_rbf_boundary(X0_low, X1_low)
    spec_high = fit_rbf_boundary(X0_high, X1_high)

    mix_low = evaluate_mixture_model(unified, spec_low, X0_low, X1_low, alpha=alpha)
    mix_high = evaluate_mixture_model(unified, spec_high, X0_high, X1_high, alpha=alpha)

    unified_low = evaluate_rbf_boundary(unified, X0_low, X1_low)
    unified_high = evaluate_rbf_boundary(unified, X0_high, X1_high)
    spec_low_eval = evaluate_rbf_boundary(spec_low, X0_low, X1_low)
    spec_high_eval = evaluate_rbf_boundary(spec_high, X0_high, X1_high)

    return {
        "unified_low": unified_low,
        "unified_high": unified_high,
        "unified_mixed": {
            "accuracy": float(np.mean([unified_low["accuracy"], unified_high["accuracy"]])),
            "overlap": float(np.mean([unified_low["overlap"], unified_high["overlap"]])),
        },
        "specialist_low": spec_low_eval,
        "specialist_high": spec_high_eval,
        "specialist_mixed": {
            "accuracy": float(np.mean([spec_low_eval["accuracy"], spec_high_eval["accuracy"]])),
            "overlap": float(np.mean([spec_low_eval["overlap"], spec_high_eval["overlap"]])),
        },
        "mixture_low": mix_low,
        "mixture_high": mix_high,
        "mixture_mixed": {
            "accuracy": float(np.mean([mix_low["accuracy"], mix_high["accuracy"]])),
            "overlap": float(np.mean([mix_low["overlap"], mix_high["overlap"]])),
        },
    }

## Bootstrap helper

In [ ]:
def bootstrap_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def summarize(vals):
    vals = np.array(vals)
    return {
        "mean": float(vals.mean()),
        "q025": float(np.quantile(vals, 0.025)),
        "q975": float(np.quantile(vals, 0.975)),
    }

def bootstrap_unified_specialist_mixture(X0_low, X1_low, X0_high, X1_high, alpha=0.5, n_boot=30, seed=9423):
    boot_rng = np.random.default_rng(seed)

    keys = [
        "unified_low", "unified_high", "unified_mixed",
        "specialist_low", "specialist_high", "specialist_mixed",
        "mixture_low", "mixture_high", "mixture_mixed",
    ]
    store = {k: [] for k in keys}

    m = min(len(X0_low), len(X1_low), len(X0_high), len(X1_high))
    X0_low = X0_low[:m]
    X1_low = X1_low[:m]
    X0_high = X0_high[:m]
    X1_high = X1_high[:m]

    for _ in range(n_boot):
        A0 = bootstrap_rows(X0_low, boot_rng)
        A1 = bootstrap_rows(X1_low, boot_rng)
        B0 = bootstrap_rows(X0_high, boot_rng)
        B1 = bootstrap_rows(X1_high, boot_rng)

        out = evaluate_unified_specialist_mixture(A0, A1, B0, B1, alpha=alpha)
        for k in keys:
            store[k].append(out[k]["overlap"])

    return {k: summarize(v) for k, v in store.items()}

## Reduced experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 140
height_block = (0, 400)
n_boot = 30
alpha = 0.5

systems = {
    "entropy": ("low_entropy", "high_entropy"),
    "unevenness": ("low_unevenness", "high_unevenness"),
}

## Base gap slices

In [ ]:
start, stop = height_block
zeta_base_gaps = zeta_gaps_full[start:stop]
poisson_base_gaps = poisson_gaps_full[start:stop]
gue_base_gaps = gue_gaps_full[:max(stop - start + 300, 900)]

len(zeta_base_gaps), len(poisson_base_gaps), len(gue_base_gaps)

## Build condition-specific feature sets

In [ ]:
def get_conditioned_features(gaps, condition_name, k=5, feature_family="minimal", n=140):
    _, stats, X = build_windows_and_features(gaps, feature_family=feature_family, k=k)
    mask = apply_condition_mask(stats, condition_name)
    Xc = X[mask]
    return Xc[:min(len(Xc), n)]

conditioned = {}

for k in window_sizes:
    conditioned[k] = {}
    for cond_lo, cond_hi in systems.values():
        for cond in [cond_lo, cond_hi]:
            conditioned[k][("zeta", cond)] = get_conditioned_features(zeta_base_gaps, cond, k=k, feature_family=feature_family, n=sample_size)
            conditioned[k][("poisson", cond)] = get_conditioned_features(poisson_base_gaps, cond, k=k, feature_family=feature_family, n=sample_size)
            conditioned[k][("gue", cond)] = get_conditioned_features(gue_base_gaps, cond, k=k, feature_family=feature_family, n=sample_size)

{k: {key: len(val) for key, val in conditioned[k].items()} for k in window_sizes}

## Main mixture-model sweep

In [ ]:
mix_results = []

for system_name, (cond_lo, cond_hi) in systems.items():
    for k in window_sizes:
        z_low = conditioned[k][("zeta", cond_lo)]
        z_high = conditioned[k][("zeta", cond_hi)]
        g_low = conditioned[k][("gue", cond_lo)]
        g_high = conditioned[k][("gue", cond_hi)]
        p_low = conditioned[k][("poisson", cond_lo)]
        p_high = conditioned[k][("poisson", cond_hi)]

        m = min(len(z_low), len(z_high), len(g_low), len(g_high), len(p_low), len(p_high), sample_size)
        if m < 40:
            continue

        zg = bootstrap_unified_specialist_mixture(z_low[:m], g_low[:m], z_high[:m], g_high[:m], alpha=alpha, n_boot=n_boot, seed=14000 + 10*k + len(system_name))
        zp = bootstrap_unified_specialist_mixture(z_low[:m], p_low[:m], z_high[:m], p_high[:m], alpha=alpha, n_boot=n_boot, seed=15000 + 10*k + len(system_name))

        row = {"system": system_name, "k": k, "sample_used": m}
        for prefix, out in [("zg", zg), ("zp", zp)]:
            for key in out:
                row[f"{prefix}_{key}_mean"] = out[key]["mean"]
                row[f"{prefix}_{key}_q025"] = out[key]["q025"]
                row[f"{prefix}_{key}_q975"] = out[key]["q975"]

        row["phase_gap_unified_low"] = row["zg_unified_low_mean"] - row["zp_unified_low_mean"]
        row["phase_gap_unified_high"] = row["zg_unified_high_mean"] - row["zp_unified_high_mean"]
        row["phase_gap_unified_mixed"] = row["zg_unified_mixed_mean"] - row["zp_unified_mixed_mean"]

        row["phase_gap_specialist_low"] = row["zg_specialist_low_mean"] - row["zp_specialist_low_mean"]
        row["phase_gap_specialist_high"] = row["zg_specialist_high_mean"] - row["zp_specialist_high_mean"]
        row["phase_gap_specialist_mixed"] = row["zg_specialist_mixed_mean"] - row["zp_specialist_mixed_mean"]

        row["phase_gap_mixture_low"] = row["zg_mixture_low_mean"] - row["zp_mixture_low_mean"]
        row["phase_gap_mixture_high"] = row["zg_mixture_high_mean"] - row["zp_mixture_high_mean"]
        row["phase_gap_mixture_mixed"] = row["zg_mixture_mixed_mean"] - row["zp_mixture_mixed_mean"]

        mix_results.append(row)

len(mix_results)

## Quick diagnostic

In [ ]:
sorted(set(r["system"] for r in mix_results)), sorted(set(r["k"] for r in mix_results))

## Mixture vs unified vs specialist phase-gap bars

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in mix_results if r["system"] == system_name]
    x = np.arange(len(window_sizes))
    width = 0.09

    metrics = [
        ("phase_gap_unified_low", "u-low"),
        ("phase_gap_specialist_low", "s-low"),
        ("phase_gap_mixture_low", "m-low"),
        ("phase_gap_unified_high", "u-high"),
        ("phase_gap_specialist_high", "s-high"),
        ("phase_gap_mixture_high", "m-high"),
        ("phase_gap_unified_mixed", "u-mixed"),
        ("phase_gap_specialist_mixed", "s-mixed"),
        ("phase_gap_mixture_mixed", "m-mixed"),
    ]

    for offset, (metric, label) in zip(np.linspace(-4, 4, len(metrics)), metrics):
        vals = [next(r for r in rows if r["k"] == k)[metric] for k in window_sizes]
        ax.bar(x + offset * width, vals, width, label=label)

    ax.set_xticks(x, window_sizes)
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("phase gap")
    ax.legend(fontsize=7, ncol=3)

plt.tight_layout()
plt.show()

## Mixture-gain heatmaps

In [ ]:
def build_gain_matrix(system_name):
    rows = [r for r in mix_results if r["system"] == system_name]
    M = np.array([
        [next(r for r in rows if r["k"] == k)["phase_gap_mixture_low"] - next(r for r in rows if r["k"] == k)["phase_gap_unified_low"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k)["phase_gap_mixture_high"] - next(r for r in rows if r["k"] == k)["phase_gap_unified_high"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k)["phase_gap_mixture_mixed"] - next(r for r in rows if r["k"] == k)["phase_gap_unified_mixed"] for k in window_sizes],
    ])
    return M

fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    M = build_gain_matrix(system_name)
    im = ax.imshow(M, aspect="auto", origin="upper")
    ax.set_title(system_name)
    ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_yticks(np.arange(3), ["low eval", "high eval", "mixed eval"])
    ax.set_xlabel("window size k")
    ax.set_ylabel("evaluation set")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="mixture gain over unified")
plt.tight_layout()
plt.show()

## Overlap comparison curves with bootstrap intervals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in mix_results if r["system"] == system_name]
    series = [
        ("zg_unified_mixed", "u-mixed"),
        ("zg_specialist_mixed", "s-mixed"),
        ("zg_mixture_mixed", "m-mixed"),
    ]

    for metric, label in series:
        means, lows, highs = [], [], []
        for k in window_sizes:
            row = next(r for r in rows if r["k"] == k)
            means.append(row[f"{metric}_mean"])
            low = row[f"{metric}_mean"] - row[f"{metric}_q025"]
            high = row[f"{metric}_q975"] - row[f"{metric}_mean"]
            lows.append(max(0.0, low))
            highs.append(max(0.0, high))
        ax.errorbar(window_sizes, means, yerr=np.vstack([np.array(lows), np.array(highs)]), marker="o", capsize=4, label=label)

    ax.set_title(f"{system_name}: zeta vs GUE overlap")
    ax.set_xlabel("window size k")
    ax.set_ylabel("overlap mean")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Compact mixture summary

In [ ]:
summary_rows = []
for system_name in ["entropy", "unevenness"]:
    for k in window_sizes:
        row = next(r for r in mix_results if r["system"] == system_name and r["k"] == k)
        summary_rows.append({
            "system": system_name,
            "k": k,
            "mixture_gain_low": float(row["phase_gap_mixture_low"] - row["phase_gap_unified_low"]),
            "mixture_gain_high": float(row["phase_gap_mixture_high"] - row["phase_gap_unified_high"]),
            "mixture_gain_mixed": float(row["phase_gap_mixture_mixed"] - row["phase_gap_unified_mixed"]),
            "mixture_minus_specialist_mixed": float(row["phase_gap_mixture_mixed"] - row["phase_gap_specialist_mixed"]),
        })
summary_rows

## Compact summary

In [ ]:
summary = {
    "window_sizes": window_sizes,
    "feature_family": feature_family,
    "sample_size_target": sample_size,
    "height_block": f"{height_block[0]+1}-{height_block[1]}",
    "alpha": alpha,
    "n_boot": n_boot,
    "systems": list(systems.keys()),
    "summary_rows": summary_rows,
}
summary

## Notes

- If the mixture model improves over unified while staying close to specialist, then the “shared core + residual correction” picture is supported.
- If the mixture model adds little beyond unified, then the shared core already captures most of the available signal.
- If mixture beats unified mainly for unevenness but not entropy, then the two condition systems differ in how much specialist correction they need.

## Next directions

A natural next notebook could do one of these:

1. learn the mixture weight alpha from the conditioning statistic instead of fixing it
2. compare multiple alpha values on the same system
3. add surrogate-aware mixture residuals
4. test the same mixture idea across feature families